In [1]:
import pandas as pd
import numpy as np

# Load the clean dataset
df = pd.read_csv('../data/cutoffs_2025_clean.csv')

print(f"Dataset loaded ✓")
print(f"Shape: {df.shape}")
print(f"\nFirst 3 rows:")
df.head(3)

Dataset loaded ✓
Shape: (11055, 6)

First 3 rows:


,college_code,college_name,branch,round,category,cutoff_rank
0,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Round3,1G,11952
1,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Round3,2AG,8509
2,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Round3,2AR,10643


In [2]:
# Quick exploration
print("=== DATASET OVERVIEW ===")
print(f"Total rows      : {len(df):,}")
print(f"Colleges        : {df['college_code'].nunique()}")
print(f"Branches        : {df['branch'].nunique()}")
print(f"Categories      : {df['category'].nunique()}")

print("\n=== AVAILABLE CATEGORIES ===")
print(sorted(df['category'].unique().tolist()))

print("\n=== TOP 10 MOST COMMON BRANCHES ===")
print(df['branch'].value_counts().head(10).to_string())

print("\n=== CUTOFF RANK RANGE ===")
print(f"Lowest  cutoff : {df['cutoff_rank'].min():,}  (most competitive)")
print(f"Highest cutoff : {df['cutoff_rank'].max():,}  (least competitive)")

=== DATASET OVERVIEW ===
Total rows      : 11,055
Colleges        : 229
Branches        : 173
Categories      : 27

=== AVAILABLE CATEGORIES ===
['1G', '1K', '1R', '2AG', '2AK', '2AR', '2BG', '2BK', '2BR', '3AG', '3AK', '3AR', '3BG', '3BK', '3BR', 'GM', 'GMK', 'GMP', 'GMR', 'OPN', 'OTH', 'SCG', 'SCK', 'SCR', 'STG', 'STK', 'STR']

=== TOP 10 MOST COMMON BRANCHES ===
branch
COMPUTER SCIENCE AND ENGINEERING                                            1902
ELECTRONICS AND COMMUNICATION ENGG                                          1199
ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING                                 768
COMPUTER                                                                     650
INFORMATION SCIENCE AND ENGINEERING                                          584
COMPUTER SCIENCE AND ENGG (ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING)     469
COMPUTER SCIENCE AND ENGINEERING (DATA SCIENCE)                              366
B TECH IN                                                 

In [3]:
# Find all fragment branch names (less than 4 words and suspiciously short)
short_branches = df[df['branch'].str.split().str.len() <= 3]['branch'].unique()
print("Suspiciously short branch names:")
for b in sorted(short_branches):
    print(f"  '{b}'")

Suspiciously short branch names:
  'AERO SPACE'
  'AERO SPACE ENGINEERING'
  'AERONAUTICAL'
  'AERONAUTICAL ENGINEERING'
  'AGRICULTURE'
  'AGRICULTURE ENGINEERING'
  'AND ROBOTIC ENGINEERING'
  'ARTIFICIAL'
  'ARTIFICIAL INTELLIGENCE'
  'AUTOMATION'
  'AUTOMATION AND ROBOTICS'
  'AUTOMOBILE'
  'AUTOMOBILE ENGINEERING'
  'AUTOMOTIVE ENGINEERING'
  'Artificial Intelligence Engg'
  'B TECH (HONS)'
  'B TECH IN'
  'B Tech in'
  'B.Plan'
  'B.TECH IN'
  'B.Tech in'
  'B.Tech in VLSI'
  'BIO-'
  'BIO- TECHNOLOGY'
  'BIO-MEDICAL ENGINEERING'
  'BIOMEDICAL'
  'BTECH IN'
  'CHEMICAL'
  'CHEMICAL ENGINEERING'
  'CIVIL'
  'CIVIL ENGINEERING'
  'CIVIL ENVIRONMENTAL ENGINEERING'
  'COMMUNICATION DESIGN'
  'COMPUTER'
  'COMPUTER ENGINEERING'
  'COMPUTER SCIENCE'
  'CYBER SECURITY'
  'DATA SCIENCES'
  'DESIGN'
  'DESIGN 5'
  'ELECTRICAL &'
  'ELECTRONICS'
  'ELECTRONICS &'
  'ELECTRONICS ENGINEERING'
  'ENGINEERING'
  'ENGINEERING & MANAGEMENT'
  'ENGINEERING DESIGN'
  'ENVIRONMENTAL ENGINEERING'
  

In [4]:
FRAGMENT_FIX = {
    # True fragments — incomplete names
    'COMPUTER'              : 'COMPUTER SCIENCE AND ENGINEERING',
    'B TECH IN'             : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING',
    'B Tech in'             : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING',
    'B TECH (HONS)'         : 'B TECH (HONS) COMPUTER SCIENCE AND ENGINEERING',
    'B.TECH IN'             : 'B.TECH IN COMPUTER SCIENCE AND ENGINEERING',
    'B.Tech in'             : 'B.TECH IN COMPUTER SCIENCE AND ENGINEERING',
    'BTECH IN'              : 'BTECH IN COMPUTER SCIENCE AND ENGINEERING',
    'ELECTRONICS &'         : 'ELECTRONICS & COMMUNICATION ENGINEERING',
    'ELECTRICAL &'          : 'ELECTRICAL & ELECTRONICS ENGINEERING',
    'ELECTRONICS'           : 'ELECTRONICS AND COMMUNICATION ENGG',
    'ROBOTICS AND'          : 'ROBOTICS AND AUTOMATION',
    'ARTIFICIAL'            : 'ARTIFICIAL INTELLIGENCE AND DATA SCIENCE',
    'ARTIFICIAL INTELLIGENCE': 'ARTIFICIAL INTELLIGENCE AND DATA SCIENCE',
    'AUTOMATION'            : 'AUTOMATION AND ROBOTICS',
    'INFORMATION'           : 'INFORMATION SCIENCE AND ENGINEERING',
    'INFORMATION SCIENCE'   : 'INFORMATION SCIENCE AND ENGINEERING',
    'MECHANICAL'            : 'MECHANICAL ENGINEERING',
    'INDUSTRIAL'            : 'INDUSTRIAL ENGINEERING & MANAGEMENT',
    'CHEMICAL'              : 'CHEMICAL ENGINEERING',
    'CIVIL'                 : 'CIVIL ENGINEERING',
    'AERONAUTICAL'          : 'AERONAUTICAL ENGINEERING',
    'AERO SPACE'            : 'AEROSPACE ENGINEERING',
    'AERO SPACE ENGINEERING': 'AEROSPACE ENGINEERING',
    'AGRICULTURE'           : 'AGRICULTURE ENGINEERING',
    'AUTOMOBILE'            : 'AUTOMOBILE ENGINEERING',
    'BIOMEDICAL'            : 'BIO-MEDICAL ENGINEERING',
    'BIO-'                  : 'BIO-TECHNOLOGY',
    'BIO- TECHNOLOGY'       : 'BIO-TECHNOLOGY',
    'MECHATRONICS'          : 'MECHATRONICS ENGINEERING',
    'MECHATRONICS 5'        : 'MECHATRONICS ENGINEERING',
    'DESIGN 5'              : 'DESIGN',
    'SCIENCE'               : 'COMPUTER SCIENCE AND ENGINEERING',
    'TECHNOLOGY'            : 'INFORMATION TECHNOLOGY',
    'ENGINEERING'           : 'COMPUTER SCIENCE AND ENGINEERING',
    'AND ROBOTIC ENGINEERING': 'ROBOTICS AND AUTOMATION',
    'SILK'                  : 'SILK TECHNOLOGY',
}

before = df['branch'].nunique()
df['branch'] = df['branch'].replace(FRAGMENT_FIX)
df['branch'] = df['branch'].str.strip()
after = df['branch'].nunique()

print(f"Branch names before: {before}")
print(f"Branch names after:  {after}")
print(f"Reduced by:          {before - after}")

print(f"\nTop 10 branches now:")
print(df['branch'].value_counts().head(10).to_string())

Branch names before: 173
Branch names after:  145
Reduced by:          28

Top 10 branches now:
branch
COMPUTER SCIENCE AND ENGINEERING                                            2595
ELECTRONICS AND COMMUNICATION ENGG                                          1405
ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING                                 768
INFORMATION SCIENCE AND ENGINEERING                                          673
B TECH IN COMPUTER SCIENCE AND ENGINEERING                                   538
ELECTRICAL & ELECTRONICS ENGINEERING                                         485
COMPUTER SCIENCE AND ENGG (ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING)     469
ARTIFICIAL INTELLIGENCE AND DATA SCIENCE                                     460
MECHANICAL ENGINEERING                                                       404
COMPUTER SCIENCE AND ENGINEERING (DATA SCIENCE)                              366


In [5]:
def recommend(rank, category, branch=None, top_n=10):
    """
    Recommend colleges based on KCET rank, category and preferred branch.
    
    Parameters:
        rank     : int   — student's KCET rank
        category : str   — e.g. 'GM', 'SCG', '2AG'
        branch   : str   — preferred branch (optional)
        top_n    : int   — number of results to show
    
    Returns:
        DataFrame of eligible colleges sorted by cutoff rank
    """
    
    # Step 1 — Filter by category
    filtered = df[df['category'] == category].copy()
    
    # Step 2 — Filter by branch if provided
    if branch:
        filtered = filtered[
            filtered['branch'].str.contains(branch, case=False, na=False)
        ]
    
    # Step 3 — Keep only colleges where student's rank <= cutoff rank
    # (cutoff rank = last rank admitted, so if your rank number is 
    #  smaller or equal, you are eligible)
    eligible = filtered[filtered['cutoff_rank'] >= rank].copy()
    
    # Step 4 — Sort by cutoff rank ascending 
    # (closest cutoff to your rank = most realistic/competitive first)
    eligible = eligible.sort_values('cutoff_rank', ascending=True)
    
    # Step 5 — Select relevant columns and return top N
    result = eligible[['college_code', 'college_name', 
                        'branch', 'category', 'cutoff_rank']].head(top_n)
    
    result = result.reset_index(drop=True)
    result.index += 1  # Start ranking from 1
    
    return result


print("recommend() function defined ✓")

recommend() function defined ✓


In [6]:
# Test 1 — Basic test
# Student with rank 12000, GM category, wants CSE
print("=" * 60)
print("TEST 1: Rank=12000 | Category=GM | Branch=CSE")
print("=" * 60)

result = recommend(rank=12000, category='GM', branch='COMPUTER SCIENCE AND ENGINEERING')
print(result.to_string())

TEST 1: Rank=12000 | Category=GM | Branch=CSE
   college_code                                                                                                                                                       college_name                                      branch category  cutoff_rank
1          E126  B M S Institute of Technology & Management, Yelahanka, Bangalore(AUTONOMOUS) POST BOX NO.6443, AVALAHALLI,DODDABALLAPURA MAIN ROAD,YELAHANKA, BANGALORE - 560064.            COMPUTER SCIENCE AND ENGINEERING       GM        12263
2          E004                              Dr. Ambedkar Institute of Technology, Bangalore(AUTONOMOUS) OUTER RING ROAD,NEAR JNANA BHARATHI CAMPUS,MALLATHAHALLI,BANGALORE-560056            COMPUTER SCIENCE AND ENGINEERING       GM        12452
3          E107                                                                      B N M Institute of Technology, Bangalore (AUTONOMOUS) 12TH MAIN, 27TH CROSS, BSK SECOND STAGE            COMPUTER SCIENCE AND ENGINEER

In [7]:
# Test 2 — Different category
print("=" * 60)
print("TEST 2: Rank=15000 | Category=2AG | Branch=CSE")
print("=" * 60)
result2 = recommend(rank=15000, category='2AG', branch='COMPUTER SCIENCE AND ENGINEERING')
print(result2.to_string())

print()

# Test 3 — No branch filter (show any branch)
print("=" * 60)
print("TEST 3: Rank=5000 | Category=GM | No branch filter")
print("=" * 60)
result3 = recommend(rank=5000, category='GM')
print(result3.to_string())

print()

# Test 4 — High rank (tough case)
print("=" * 60)
print("TEST 4: Rank=50000 | Category=GM | Branch=CSE")
print("=" * 60)
result4 = recommend(rank=50000, category='GM', branch='COMPUTER SCIENCE AND ENGINEERING')
print(result4.to_string())

TEST 2: Rank=15000 | Category=2AG | Branch=CSE
   college_code                                                                                                                                                       college_name                                      branch category  cutoff_rank
1          E178                                                The National Institute of Engineering(NORTH CAMPUS), No.50(Part) Koorgalli,Hootgalli Industrial Area ,Mysuru-570018            COMPUTER SCIENCE AND ENGINEERING      2AG        16250
2          E004                              Dr. Ambedkar Institute of Technology, Bangalore(AUTONOMOUS) OUTER RING ROAD,NEAR JNANA BHARATHI CAMPUS,MALLATHAHALLI,BANGALORE-560056            COMPUTER SCIENCE AND ENGINEERING      2AG        16325
3          E285                                                                                                        RV University R.V. VIDYANIKETAN POST, MYSORE ROAD,BANGALORE  B TECH IN COMPUTER SCIENCE AND ENGINEE

In [8]:
# Improved recommend() — removes duplicate colleges
def recommend(rank, category, branch=None, top_n=10):
    
    # Step 1 — Filter by category
    filtered = df[df['category'] == category].copy()
    
    # Step 2 — Filter by branch if provided
    if branch:
        filtered = filtered[
            filtered['branch'].str.contains(branch, case=False, na=False)
        ]
    
    # Step 3 — Keep only eligible colleges
    eligible = filtered[filtered['cutoff_rank'] >= rank].copy()
    
    # Step 4 — If same college appears multiple times, keep the one
    # with cutoff rank closest to student's rank (most realistic)
    eligible = eligible.sort_values('cutoff_rank', ascending=True)
    eligible = eligible.drop_duplicates(
        subset=['college_code', 'branch'], keep='first'
    )
    
    # Step 5 — Clean up college name (remove address)
    eligible['college_name'] = eligible['college_name'].str.split(
        r'[\(\,]', regex=True
    ).str[0].str.strip()
    
    # Step 6 — Return top N
    result = eligible[['college_code', 'college_name', 
                        'branch', 'category', 'cutoff_rank']].head(top_n)
    result = result.reset_index(drop=True)
    result.index += 1
    
    return result

# Retest
print("=" * 60)
print("TEST 1 IMPROVED: Rank=12000 | Category=GM | Branch=CSE")
print("=" * 60)
print(recommend(rank=12000, category='GM', 
      branch='COMPUTER SCIENCE AND ENGINEERING').to_string())

TEST 1 IMPROVED: Rank=12000 | Category=GM | Branch=CSE
   college_code                                                     college_name                                           branch category  cutoff_rank
1          E126                       B M S Institute of Technology & Management                 COMPUTER SCIENCE AND ENGINEERING       GM        12263
2          E004                             Dr. Ambedkar Institute of Technology                 COMPUTER SCIENCE AND ENGINEERING       GM        12452
3          E107                                    B N M Institute of Technology                 COMPUTER SCIENCE AND ENGINEERING       GM        14324
4          E118                                      RNS Institute of Technology                 COMPUTER SCIENCE AND ENGINEERING       GM        14842
5          E097                                    C M R Institute of Technology                 COMPUTER SCIENCE AND ENGINEERING       GM        16221
6          E037                  

In [9]:
def show_recommendations(rank, category, branch=None, top_n=10):
    
    print(f"\n{'='*60}")
    print(f"  KCET COLLEGE RECOMMENDATIONS")
    print(f"{'='*60}")
    print(f"  Rank     : {rank:,}")
    print(f"  Category : {category}")
    print(f"  Branch   : {branch if branch else 'Any'}")
    print(f"{'='*60}\n")
    
    result = recommend(rank, category, branch, top_n)
    
    if len(result) == 0:
        print("❌ No colleges found for this rank and category.")
        print("   Try a higher rank or different category.")
        return
    
    for i, row in result.iterrows():
        print(f"  {i}. {row['college_name']}")
        print(f"     Branch  : {row['branch']}")
        print(f"     Cutoff  : {row['cutoff_rank']:,}")
        print()
    
    print(f"  Total colleges found: {len(result)}")
    print(f"{'='*60}\n")

# Final test
show_recommendations(rank=12000, category='GM', 
                     branch='COMPUTER SCIENCE AND ENGINEERING')


  KCET COLLEGE RECOMMENDATIONS
  Rank     : 12,000
  Category : GM
  Branch   : COMPUTER SCIENCE AND ENGINEERING

  1. B M S Institute of Technology & Management
     Branch  : COMPUTER SCIENCE AND ENGINEERING
     Cutoff  : 12,263

  2. Dr. Ambedkar Institute of Technology
     Branch  : COMPUTER SCIENCE AND ENGINEERING
     Cutoff  : 12,452

  3. B N M Institute of Technology
     Branch  : COMPUTER SCIENCE AND ENGINEERING
     Cutoff  : 14,324

  4. RNS Institute of Technology
     Branch  : COMPUTER SCIENCE AND ENGINEERING
     Cutoff  : 14,842

  5. C M R Institute of Technology
     Branch  : COMPUTER SCIENCE AND ENGINEERING
     Cutoff  : 16,221

  6. K.L.S. Gogte Institute of Technology
     Branch  : COMPUTER SCIENCE AND ENGINEERING
     Cutoff  : 16,368

  7. SDM College of Engineering
     Branch  : COMPUTER SCIENCE AND ENGINEERING
     Cutoff  : 16,711

  8. New Horizon College of Engineering
     Branch  : COMPUTER SCIENCE AND ENGINEERING
     Cutoff  : 17,363

  9. M . S